In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, IntegerType

# Sample data
rows = [
    Row(
        id=1,
        names=["Alice", "Bob"],
        scores=[85, 90],
        info=Row(age=25, city="New York"),
        json_str='{"dept": "Engineering", "level": 2}'
    ),
    Row(
        id=2,
        names=["Carol"],
        scores=[78, 88, 92],
        info=Row(age=30, city="San Francisco"),
        json_str='{"dept": "HR", "level": 1}'
    )
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("names", ArrayType(StringType()), True),
    StructField("scores", ArrayType(IntegerType()), True),
    StructField("info", StructType([
        StructField("age", IntegerType(), True),
        StructField("city", StringType(), True)
    ]), True),
    StructField("json_str", StringType(), True)
])

df = spark.createDataFrame(rows, schema)
df.createOrReplaceTempView("df")
display(df)

In [0]:
from pyspark.sql import functions as F

df.withColumn("names_length", F.size(F.col("names"))).display()

In [0]:
from pyspark.sql import functions as F

df.withColumn("names_length", F.array_contains(F.col("names"), "Alice")).display()

In [0]:
from pyspark.sql import functions as F

df.withColumn("city_words", F.split(F.col("info.city"), " ")).display()

In [0]:
%sql
SELECT
  *,
  size(names) AS names_length,
  array_contains(names, 'Alice') AS names_has_alice,
  split(info.city, ' ') AS city_words
FROM df


In [0]:
from pyspark.sql import functions as F

df.select("id", F.explode(F.col("names").alias("name"))).display()

In [0]:
%sql
select id, explode(names) as name
from df;

In [0]:
df.withColumn("age", F.col("info.age")).alias("age")\
    .withColumn("city", F.col("info.city")).alias("city").display()

In [0]:
%sql
select *, info.age as age, info.city as city
from df

In [0]:
df.withColumn("dept", F.get_json_object(F.col("json_str"), "$.dept"))\
    .withColumn("level", F.get_json_object(F.col("json_str"), "$.level"))\
    .display()

In [0]:
%sql
SELECT
  id,
  get_json_object(json_str, '$.dept') AS dept,
  get_json_object(json_str, '$.level') AS level
FROM df;

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


# Parse JSON string into struct
json_schema = StructType([
    StructField("dept", StringType(), True),
    StructField("level", IntegerType(), True)
])
from_json_df = df.withColumn("json_struct", F.from_json(F.col("json_str"), json_schema))
# Accessing fields in the 'info' struct
struct_df = from_json_df.withColumn("dept", F.col("json_struct.dept"))

display(struct_df)

In [0]:
%sql
SELECT *,
from_json(json_str, 'STRUCT<dept:STRING,level:INT>') AS json_struct,
from_json(json_str, 'STRUCT<dept:STRING,level:INT>').dept AS dept
FROM df